In [ ]:
print("hii Yuvi")

hii Yuvi


Weeks 1–2:

  • Set up environment and install dependencies.

  • Implement file upload and multi-format text extraction.


  • Milestone 1 (Week 2):
File upload and accurate text extraction operational.



In [ ]:

!pip install PyPDF2 pdfplumber python-docx pytesseract pillow pdf2image
!apt-get install -y poppler-utils


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
poppler-utils is already the newest version (22.02.0-2ubuntu0.11).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [ ]:
import os
from google.colab import files
import docx
import PyPDF2
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image



In [ ]:
# TXT
def extract_from_txt(path):
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        return f.read()

# DOCX
def extract_from_docx(path):
    doc = docx.Document(path)
    return "\n".join([p.text for p in doc.paragraphs])

# PDF (pdfplumber)
def extract_from_pdf_pdfplumber(path):
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() or ""
    return text

# PDF (PyPDF2)
def extract_from_pdf_pypdf2(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

# PDF (OCR using pytesseract)
def extract_from_pdf_ocr(path):
    pages = convert_from_path(path)
    text = ""
    for page in pages:
        text += pytesseract.image_to_string(page)
    return text

# SMART PDF extractor (tries all)
def extract_from_pdf(path):
    text = extract_from_pdf_pdfplumber(path)
    if not text.strip():
        text = extract_from_pdf_pypdf2(path)
    if not text.strip():
        text = extract_from_pdf_ocr(path)
    return text

# AUTO extractor for any file
def extract_auto(path):
    ext = os.path.splitext(path)[-1].lower()
    if ext == ".txt":
        return extract_from_txt(path)
    elif ext == ".docx":
        return extract_from_docx(path)
    elif ext == ".pdf":
        return extract_from_pdf(path)
    else:
        return f"❌ Unsupported file type: {ext}"


In [ ]:

uploaded = files.upload()

for filename in uploaded.keys():
    print(f"📂 File uploaded: {filename}")
    text = extract_auto(filename)

    print(f"\n📖 Content from {filename}:\n")
    print(text[:1500])   # show first 1500 characters as preview


Saving CS_2026_Syllabus.pdf to CS_2026_Syllabus (2).pdf
📂 File uploaded: CS_2026_Syllabus (2).pdf

📖 Content from CS_2026_Syllabus (2).pdf:

GATE 2026 IIT Guwahati | Organizing Institute

 

cs Comput cience and Information Technology

Section 1: Engineering Mathematics

Discrete Mathematics: Propositional and first order logic. Sets, relations, functions, partial
orders and lattices. Monoids, Groups. Graphs: connectivity, matching, colouring.
Combinatorics: counting, recurrence relations, generating functions.

Linear Algebra: Matrices, determinants, system of linear equations, eigenvalues and
eigenvectors, LU decomposition.

Calculus: Limits, continuity and differentiability, Maxima and minima, Mean value theorem,
Integration.

Probability and Statistics: Random variables, Uniform, normal, exponential, Poisson
and binomial distributions. Mean, median, mode and standard deviation. Conditional
probability and Bayes theorem.

Section 2: Digital Logic

Boolean algebra. Combinational and 

In [ ]:
output_file = "extracted_output.txt"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(text)

print(f"✅ Extracted text saved to {output_file}")
files.download(output_file)


✅ Extracted text saved to extracted_output.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Weeks 3–4:

• Integrate LLM for audiobook-style text rewriting.

• Build API connection between Streamlit and backend LLM processing.

• Milestone 2 (Week 4):LLM-based text rewriting working and demonstrably improving narration.

In [36]:
#Week 3–4 — LLM Integration and Text Rewriting
#Install dependencies

!pip install PyPDF2 pdfplumber python-docx openai


In [ ]:
# Import necessary modules
from google.colab import files
import os, pdfplumber, docx
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI("<YOUR_API_KEY_HERE>")

print(" Libraries imported and API initialized successfully")



 Libraries imported and API initialized successfully


In [38]:
# --- Function: Detect file type and extract text ---
def extract_text_auto(filepath):
    ext = os.path.splitext(filepath)[1].lower()

    if ext == ".txt":
        with open(filepath, "r", encoding="utf-8") as f:
            return f.read()

    elif ext == ".docx":
        doc = docx.Document(filepath)
        return "\n".join([para.text for para in doc.paragraphs])

    elif ext == ".pdf":
        text = ""
        with pdfplumber.open(filepath) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
        return text

    else:
        raise ValueError("❌ Unsupported file type")

# --- Upload files ---
uploaded = files.upload()

# --- Extract text from uploaded file(s) ---
for filename in uploaded.keys():
    extracted_text = extract_text_auto(filename)
    print(f"\n📘 {filename}: extracted {len(extracted_text)} characters.")
    print(extracted_text[:500])  # preview first 500 chars


Saving example.txt.txt to example.txt (2).txt

📘 example.txt (2).txt: extracted 421 characters.
It's a txt file hay there I am yuvi.

My name is Yuvraj Sahoo, and I am currently pursuing a B.Tech in Computer Science and Engineering at BPUT University. I am in my 3rd year (5th semester) with a strong academic background, maintaining an overall CGPA of around 8.5 in the first four semesters. I am passionate about technology and eager to learn new skills that will help me grow both academically and professionally.



In [ ]:
# --- Function: Rewrite text with GPT ---
def rewrite_text_with_llm(text, style="audiobook", tone="conversational"):
    """Rewrites text into audiobook-friendly narration"""
    if not text.strip():
        return "⚠️ No text found to rewrite."

    prompt = f"""
    Rewrite the following text in a {style} style
    using a {tone} tone. Make it engaging and natural for listening:

    {text[:1500]}  # keep input within safe limit
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are an expert audiobook narrator."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"❌ LLM error: {e}"


In [ ]:
# --- Process uploaded file(s) ---
for filename in uploaded.keys():
    raw_text = extract_text_auto(filename)
    rewritten = rewrite_text_with_llm(raw_text)

    # Save rewritten version
    out_file = f"{os.path.splitext(filename)[0]}_rewritten.txt"
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(rewritten)

    print(f"\n✅ Rewritten text saved as: {out_file}")
    print("\n🔊 Preview:\n", rewritten[:600])



✅ Rewritten text saved as: example.txt (1)_rewritten.txt

🔊 Preview:
 Hey there! I’m Yuvraj Sahoo, but you can call me Yuvi. Right now, I’m diving into my third year of a B.Tech in Computer Science and Engineering at BPUT University. Can you believe I’m already in my fifth semester? Time flies! 

I’ve been fortunate to maintain a solid academic record, holding an overall CGPA of around 8.5 through my first four semesters. I have a big passion for technology, and I'm always on the lookout for new skills to learn. My goal? To grow both academically and professionally. So, let’s see where this journey takes me!


Weeks 5–6:

• Integrate and test open-source TTS conversion.

• Ensure support for different voice options and error handling.

• Milestone 3 (Week 6):
Audio file generation (from rewritten text) stable and high-quality.

In [ ]:
#Week 5–6 — TTS Conversion + Voice Options

# Install TTS and audio libraries
!pip install gTTS pydub


In [ ]:
#Import libraries

from gtts import gTTS
from IPython.display import Audio
from pydub import AudioSegment
import os


In [ ]:
# Basic test for TTS conversion
text = "Hello everyone! This is our first Text to Speech test for the audiobook project."
tts = gTTS(text=text, lang='en')
tts.save("week5_test.mp3")

print("✅ Basic audio generated successfully!")
Audio("week5_test.mp3")


✅ Basic audio generated successfully!


In [ ]:
#Handle long text by splitting into chunks
def split_text(text, max_chars=200):
    """Split long text into smaller pieces (so gTTS can process it)"""
    words = text.split()
    chunks, chunk, length = [], [], 0
    for w in words:
        if length + len(w) + 1 > max_chars:
            chunks.append(" ".join(chunk))
            chunk, length = [], 0
        chunk.append(w)
        length += len(w) + 1
    if chunk:
        chunks.append(" ".join(chunk))
    return chunks

# Example
sample_long_text = """
Artificial Intelligence is transforming industries around the world.
It helps in healthcare, finance, and education by making intelligent decisions.
"""
print("🔹 Sample split:", split_text(sample_long_text))


🔹 Sample split: ['Artificial Intelligence is transforming industries around the world. It helps in healthcare, finance, and education by making intelligent decisions.']


In [ ]:
#Generate audio for all chunks and merge
def text_to_speech_full(text, filename="audiobook_output.mp3"):
    """Converts long text into a single audiobook file"""
    chunks = split_text(text, max_chars=200)
    combined = AudioSegment.empty()
    temp_files = []

    for i, chunk in enumerate(chunks):
        tts = gTTS(text=chunk, lang='en')
        fname = f"temp_chunk_{i}.mp3"
        tts.save(fname)
        temp_files.append(fname)
        combined += AudioSegment.from_mp3(fname)

    combined.export(filename, format="mp3")

    # Cleanup temp files
    for f in temp_files:
        os.remove(f)

    print(f"✅ Final audiobook saved as: {filename}")
    return filename

# Example usage (use rewritten text from Week 4)
audiobook_file = text_to_speech_full(sample_long_text)
Audio(audiobook_file)


✅ Final audiobook saved as: audiobook_output.mp3


In [ ]:
#Add multiple voice variants (US/UK/AUS)
def generate_voice_variants(text, lang="en"):
    """Generate multiple accent variants"""
    variants = {
        "US_English": {"lang": "en", "tld": "com"},
        "UK_English": {"lang": "en", "tld": "co.uk"},
        "Australian_English": {"lang": "en", "tld": "com.au"}
    }

    outputs = {}
    for name, opts in variants.items():
        try:
            tts = gTTS(text=text[:400], lang=opts["lang"], tld=opts["tld"])
            fname = f"voice_{name}.mp3"
            tts.save(fname)
            outputs[name] = fname
            print(f"✅ Generated {name}")
        except Exception as e:
            print(f"❌ Error for {name}: {e}")

    return outputs

# Test it
voices = generate_voice_variants("Hay there i am yuvraj it's a demo!")
for name, file in voices.items():
    print(f"\n🎙️ {name}:")
    display(Audio(file))


✅ Generated US_English
✅ Generated UK_English
✅ Generated Australian_English

🎙️ US_English:



🎙️ UK_English:



🎙️ Australian_English:


In [ ]:
#Combine everything (final pipeline)
def final_tts_pipeline(rewritten_text):
    """Full pipeline: convert rewritten text → audiobook"""
    try:
        final_audio = text_to_speech_full(rewritten_text, filename="week6_final_audiobook.mp3")
        print("🎧 Final audiobook ready!")
        return Audio(final_audio)
    except Exception as e:
        print("❌ Error in final pipeline:", e)

# Example (use your rewritten LLM text from Week 4)
final_tts_pipeline("This is a demonstration of the final audiobook generation pipeline for Week 6.")


✅ Final audiobook saved as: week6_final_audiobook.mp3
🎧 Final audiobook ready!


Weeks 7–8:

• Finalize UI/UX in Streamlit.

• Conduct thorough testing, optimize performance, and complete documentation.

• Milestone 4 (Week 8):
Full application workflow—document upload to audio download—operational, user
friendly, and documented.



In [ ]:
%%writefile app.py
# ==============================
# 🎧 Week 7–8: AI Audiobook Generator (Streamlit Frontend)
# ==============================

# Step 1️⃣ Import Libraries
import streamlit as st
import pdfplumber, docx, os
from gtts import gTTS
from io import BytesIO
from openai import OpenAI

# Step 2️⃣ Initialize OpenAI Client
client = OpenAI("<Your_API_Key_Here>")

# Step 3️⃣ Streamlit UI Setup
st.set_page_config(page_title="AI Audiobook Generator", layout="centered")
st.title("🎧 AI Audiobook Generator")
st.write("Upload a document → Extract text → Rewrite with AI → Listen to it as an audiobook!")

uploaded_file = st.file_uploader("📤 Upload your file (.pdf or .docx)", type=["pdf", "docx"])

# Step 4️⃣ Text Extraction
if uploaded_file:
    ext = os.path.splitext(uploaded_file.name)[-1].lower()
    text = ""

    if ext == ".pdf":
        with pdfplumber.open(uploaded_file) as pdf:
            for page in pdf.pages:
                text += page.extract_text() or ""
    elif ext == ".docx":
        doc = docx.Document(uploaded_file)
        for para in doc.paragraphs:
            text += para.text + "\n"

    st.subheader("📄 Extracted Text")
    st.text_area("Extracted Content", text[:1000] + "..." if len(text) > 1000 else text, height=200)

    # Step 5️⃣ Rewrite Text with OpenAI LLM
    if st.button("✨ Rewrite with AI"):
        with st.spinner("Rewriting using AI..."):
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "You are an audiobook narrator. Rewrite this text in a clear, engaging, natural audiobook tone."},
                    {"role": "user", "content": text[:2000]}  # limit text for safety
                ]
            )
            rewritten = response.choices[0].message.content

            st.subheader("🧠 Rewritten Text")
            st.write(rewritten)

            # Step 6️⃣ Text-to-Speech (TTS) Conversion
            if st.button("🔊 Convert to Audio"):
                tts = gTTS(text=rewritten, lang="en")
                audio_bytes = BytesIO()
                tts.write_to_fp(audio_bytes)
                audio_bytes.seek(0)

                st.audio(audio_bytes, format="audio/mp3")
                st.success("✅ Audiobook generated successfully!")


Writing app.py


In [ ]:
!pip install streamlit pyngrok pdfplumber python-docx gTTS openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 40.3 MB/s eta 0:00:00


In [ ]:
#33vsfzM16MViUaq0y6CNEGO3tdh_437PSr1NzzsG8Q8cpQkav

In [ ]:
!ngrok config add-authtoken 33vsfzM16MViUaq0y6CNEGO3tdh_437PSr1NzzsG8Q8cpQkav


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print("🌍 Streamlit App URL:", public_url)

!streamlit run app.py --server.port 8501


🌍 Streamlit App URL: NgrokTunnel: "https://nonideational-vertically-billi.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.91.164.59:8501

  Stopping...


In [ ]:
!pkill streamlit
!pkill ngrok




In [ ]:
from pyngrok import ngrok
import time

public_url = ngrok.connect(8501)
print("🌍 Open this link in a NEW browser tab:", public_url)

time.sleep(3)
!streamlit run app.py --server.port 8501 &>/dev/null &



🌍 Open this link in a NEW browser tab: NgrokTunnel: "https://nonideational-vertically-billi.ngrok-free.dev" -> "http://localhost:8501"
